# Week 10 — Evaluation Report

Notebook này tổng hợp kết quả đánh giá chi tiết cho tất cả các models đã train/fine-tune trong dự án:

| Model | Nguồn | Checkpoint |
|-------|-------|------------|
| AlexNet (from scratch) | `week67_alexnet-from-scratch.ipynb` | `alexnet_from_scratch_best.pt` |
| ResNet50 (transfer) | `week89_transfer_learning.ipynb` | `resnet50_best.pt` |
| MobileNetV2 (transfer) | `week89_transfer_learning.ipynb` | `mobilenet_v2_best.pt` |
| EfficientNet-B0 (transfer) | `week89_transfer_learning.ipynb` | `efficientnet_b0_best.pt` |

Với mỗi model, ta đánh giá trên **train / val / test** và báo cáo:

- **Accuracy** (top-1)
- **Precision** (macro & weighted)
- **Recall** (macro & weighted)
- **F1-score** (macro & weighted)
- **Confusion matrix**
- **Per-class classification report** (top confused pairs)

In [ ]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
ARTIFACTS_DIR = Path("./artifacts/datasets")
CHECKPOINT_DIR = Path("./artifacts/checkpoints")
HISTORY_DIR = Path("./artifacts/training")

CLASS_TO_IDX_PATH = ARTIFACTS_DIR / "class_to_idx.json"
TRAIN_RECORDS_PATH = ARTIFACTS_DIR / "train_records.json"
VAL_RECORDS_PATH = ARTIFACTS_DIR / "val_records.json"
TEST_RECORDS_PATH = ARTIFACTS_DIR / "test_records.json"

IMAGE_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 64
NUM_WORKERS = 0

USE_BBOX = False
BBOX_PADDING = 0.05

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device     : {DEVICE}")
print(f"Batch size : {BATCH_SIZE}")

## 1. Utilities và Dataset

In [ ]:
def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}.")
    return json.loads(path.read_text(encoding="utf-8"))


def parse_annotation(annotation_path: str) -> dict:
    root = ET.parse(annotation_path).getroot()
    size_node = root.find("size")
    object_node = root.find("object")
    bbox_node = object_node.find("bndbox")
    return {
        "width": int(size_node.findtext("width")),
        "height": int(size_node.findtext("height")),
        "bbox": {
            "xmin": int(bbox_node.findtext("xmin")),
            "ymin": int(bbox_node.findtext("ymin")),
            "xmax": int(bbox_node.findtext("xmax")),
            "ymax": int(bbox_node.findtext("ymax")),
        },
    }


def crop_with_bbox(
    image: Image.Image, bbox: dict, padding_ratio: float = 0.0
) -> Image.Image:
    xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
    bw, bh = xmax - xmin, ymax - ymin
    px, py = int(bw * padding_ratio), int(bh * padding_ratio)
    xmin = max(0, xmin - px)
    ymin = max(0, ymin - py)
    xmax = min(image.width, xmax + px)
    ymax = min(image.height, ymax + py)
    return image.crop((xmin, ymin, xmax, ymax))


class StanfordDogsDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        transform=None,
        use_bbox: bool = False,
        bbox_padding: float = 0.0,
    ) -> None:
        self.records = list(records)
        self.transform = transform
        self.use_bbox = use_bbox
        self.bbox_padding = bbox_padding

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        with Image.open(record["image_path"]) as pil_image:
            image = pil_image.convert("RGB")
        if self.use_bbox:
            annotation = parse_annotation(record["annotation_path"])
            image = crop_with_bbox(image, annotation["bbox"], padding_ratio=self.bbox_padding)
        if self.transform is not None:
            image = self.transform(image)
        return image, record["class_id"]

In [ ]:
class_to_idx = load_json(CLASS_TO_IDX_PATH)
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

train_records = load_json(TRAIN_RECORDS_PATH)
val_records = load_json(VAL_RECORDS_PATH)
test_records = load_json(TEST_RECORDS_PATH)

num_classes = len(class_to_idx)

breed_names = [
    idx_to_class[i].split("-", 1)[1].replace("_", " ") for i in range(num_classes)
]

print(f"Num classes   : {num_classes}")
print(f"Train samples : {len(train_records)}")
print(f"Val samples   : {len(val_records)}")
print(f"Test samples  : {len(test_records)}")

In [ ]:
eval_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

split_configs = {
    "train": train_records,
    "val": val_records,
    "test": test_records,
}

data_loaders: dict[str, DataLoader] = {}
for split_name, records in split_configs.items():
    ds = StanfordDogsDataset(
        records, transform=eval_transform, use_bbox=USE_BBOX, bbox_padding=BBOX_PADDING
    )
    data_loaders[split_name] = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    print(f"{split_name:5s} loader: {len(ds)} samples, {len(data_loaders[split_name])} batches")

## 2. Định nghĩa models và registry

Tái tạo kiến trúc của từng model giống hệt lúc train để load `state_dict` tương thích.

In [ ]:
class AlexNetFromScratch(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.5) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def _build_transfer_model(backbone_name: str, num_classes: int) -> nn.Module:
    """Rebuild transfer-learning architecture (matching week89 exactly)."""
    dropout = 0.3
    if backbone_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(2048, num_classes))
    elif backbone_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(1280, num_classes))
    elif backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(1280, num_classes))
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")
    return model


MODEL_REGISTRY: dict[str, dict] = {
    "AlexNet (scratch)": {
        "builder": lambda nc: AlexNetFromScratch(nc, dropout=0.5),
        "checkpoint": CHECKPOINT_DIR / "alexnet_from_scratch_best.pt",
        "source": "week67",
    },
    "ResNet50 (transfer)": {
        "builder": lambda nc: _build_transfer_model("resnet50", nc),
        "checkpoint": CHECKPOINT_DIR / "resnet50_best.pt",
        "source": "week89",
    },
    "MobileNetV2 (transfer)": {
        "builder": lambda nc: _build_transfer_model("mobilenet_v2", nc),
        "checkpoint": CHECKPOINT_DIR / "mobilenet_v2_best.pt",
        "source": "week89",
    },
    "EfficientNet-B0 (transfer)": {
        "builder": lambda nc: _build_transfer_model("efficientnet_b0", nc),
        "checkpoint": CHECKPOINT_DIR / "efficientnet_b0_best.pt",
        "source": "week89",
    },
}

print("Model registry:")
for name, cfg in MODEL_REGISTRY.items():
    exists = cfg["checkpoint"].exists()
    status = "FOUND" if exists else "NOT FOUND"
    print(f"  {name:30s} | {cfg['checkpoint'].name:40s} | {status}")

## 3. Hàm đánh giá chính

Hàm `collect_predictions` chạy inference trên toàn bộ loader và trả về `(all_labels, all_preds)` để tính metrics với sklearn.

In [ ]:
@torch.inference_mode()
def collect_predictions(
    model: nn.Module, loader: DataLoader, device: torch.device
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_labels, all_preds = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_labels.append(labels.numpy())
        all_preds.append(preds)
    return np.concatenate(all_labels), np.concatenate(all_preds)


def compute_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, num_classes: int
) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(num_classes))),
    }


def print_split_metrics(metrics: dict, split_name: str) -> None:
    print(f"  {split_name.upper():5s} | "
          f"Acc={metrics['accuracy']:.4f} | "
          f"P(macro)={metrics['precision_macro']:.4f} P(wt)={metrics['precision_weighted']:.4f} | "
          f"R(macro)={metrics['recall_macro']:.4f} R(wt)={metrics['recall_weighted']:.4f} | "
          f"F1(macro)={metrics['f1_macro']:.4f} F1(wt)={metrics['f1_weighted']:.4f}")

## 4. Đánh giá tất cả các models

Với mỗi model có checkpoint, ta:
1. Load model + `state_dict`
2. Chạy inference trên train / val / test
3. Tính toàn bộ metrics và lưu lại

In [ ]:
all_eval_results: dict[str, dict] = {}

for model_name, cfg in MODEL_REGISTRY.items():
    ckpt_path = cfg["checkpoint"]
    if not ckpt_path.exists():
        print(f"\n SKIP {model_name} — checkpoint not found: {ckpt_path.name}")
        continue

    print(f"\n{'='*70}")
    print(f"  {model_name}")
    print(f"{'='*70}")

    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = cfg["builder"](num_classes)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    best_epoch = checkpoint.get("epoch", "?")
    best_val_top1 = checkpoint.get("best_val_top1", None)

    print(f"  Params     : {total_params:,}")
    print(f"  Best epoch : {best_epoch}")
    if best_val_top1 is not None:
        print(f"  Best val@1 : {best_val_top1:.4f}")

    model_results = {"params": total_params, "best_epoch": best_epoch}

    for split_name, loader in data_loaders.items():
        y_true, y_pred = collect_predictions(model, loader, DEVICE)
        metrics = compute_metrics(y_true, y_pred, num_classes)
        model_results[split_name] = metrics
        model_results[f"{split_name}_y_true"] = y_true
        model_results[f"{split_name}_y_pred"] = y_pred
        print_split_metrics(metrics, split_name)

    all_eval_results[model_name] = model_results

    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n\nEvaluated {len(all_eval_results)} model(s).")

## 5. Bảng tổng hợp so sánh

In [ ]:
if not all_eval_results:
    print("Không có model nào được đánh giá. Hãy chạy train trước.")
else:
    header = (
        f"{'Model':30s} | {'Params':>10s} | {'Split':5s} | "
        f"{'Acc':>7s} | {'P(macro)':>8s} | {'P(wt)':>8s} | "
        f"{'R(macro)':>8s} | {'R(wt)':>8s} | {'F1(macro)':>9s} | {'F1(wt)':>8s}"
    )
    sep = "-" * len(header)
    print(header)
    print(sep)

    for model_name, results in all_eval_results.items():
        params_str = f"{results['params']/1e6:.1f}M"
        for split_name in ["train", "val", "test"]:
            m = results[split_name]
            print(
                f"{model_name:30s} | {params_str:>10s} | {split_name:5s} | "
                f"{m['accuracy']:7.4f} | {m['precision_macro']:8.4f} | {m['precision_weighted']:8.4f} | "
                f"{m['recall_macro']:8.4f} | {m['recall_weighted']:8.4f} | {m['f1_macro']:9.4f} | {m['f1_weighted']:8.4f}"
            )
            params_str = ""
        print(sep)

## 6. Classification Report chi tiết (test set)

In `classification_report` của sklearn cho mỗi model trên tập **test**.

In [ ]:
for model_name, results in all_eval_results.items():
    print(f"\n{'='*70}")
    print(f"  Classification Report — {model_name} (TEST set)")
    print(f"{'='*70}\n")
    y_true = results["test_y_true"]
    y_pred = results["test_y_pred"]
    report = classification_report(
        y_true,
        y_pred,
        target_names=breed_names,
        zero_division=0,
        digits=4,
    )
    print(report)

## 7. Confusion Matrix (test set)

Vẽ confusion matrix dưới dạng heatmap cho từng model. Vì có 120 classes nên ta vẽ:
- **Full confusion matrix** (nhỏ, overview)
- **Top-20 most confused class pairs** (dễ đọc hơn)

In [ ]:
def plot_confusion_matrix_full(
    cm: np.ndarray, model_name: str, split: str = "test"
) -> None:
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"Confusion Matrix — {model_name} ({split})", fontsize=14, pad=12)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def plot_top_confused_pairs(
    cm: np.ndarray,
    class_names: list[str],
    model_name: str,
    top_k: int = 20,
    split: str = "test",
) -> None:
    np.fill_diagonal(cm, 0)
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(n):
            if cm[i, j] > 0:
                pairs.append((cm[i, j], i, j))
    pairs.sort(reverse=True)
    pairs = pairs[:top_k]

    if not pairs:
        print(f"  No off-diagonal errors for {model_name} ({split})")
        return

    labels = [f"{class_names[i]}\n\u2192 {class_names[j]}" for _, i, j in pairs]
    counts = [c for c, _, _ in pairs]

    fig, ax = plt.subplots(figsize=(12, max(4, top_k * 0.35)))
    y_pos = np.arange(len(labels))
    ax.barh(y_pos, counts, color="#c44e52", edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Number of misclassifications")
    ax.set_title(f"Top-{top_k} Confused Pairs — {model_name} ({split})", fontsize=13)
    for idx, count in enumerate(counts):
        ax.text(count + 0.3, idx, str(count), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


for model_name, results in all_eval_results.items():
    cm = results["test"]["confusion_matrix"]
    plot_confusion_matrix_full(cm, model_name, split="test")
    plot_top_confused_pairs(cm.copy(), breed_names, model_name, top_k=20, split="test")

## 8. So sánh accuracy trên cả 3 tập (bar chart)

In [ ]:
if all_eval_results:
    model_names = list(all_eval_results.keys())
    splits = ["train", "val", "test"]
    split_colors = {"train": "#4c72b0", "val": "#dd8452", "test": "#55a868"}

    x = np.arange(len(model_names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(max(10, len(model_names) * 3), 6))

    for i, split in enumerate(splits):
        accs = [all_eval_results[m][split]["accuracy"] * 100 for m in model_names]
        bars = ax.bar(x + i * width, accs, width, label=split, color=split_colors[split])
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}",
                ha="center", va="bottom", fontsize=9,
            )

    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy Comparison — All Models \u00d7 All Splits", fontsize=14)
    ax.set_xticks(x + width)
    ax.set_xticklabels(model_names, fontsize=10)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9. So sánh F1-score (macro) trên cả 3 tập

In [ ]:
if all_eval_results:
    fig, ax = plt.subplots(figsize=(max(10, len(model_names) * 3), 6))

    for i, split in enumerate(splits):
        f1s = [all_eval_results[m][split]["f1_macro"] * 100 for m in model_names]
        bars = ax.bar(x + i * width, f1s, width, label=split, color=split_colors[split])
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}",
                ha="center", va="bottom", fontsize=9,
            )

    ax.set_ylabel("F1-score macro (%)")
    ax.set_title("F1 (macro) Comparison — All Models \u00d7 All Splits", fontsize=14)
    ax.set_xticks(x + width)
    ax.set_xticklabels(model_names, fontsize=10)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Per-class F1 distribution (test set)

Boxplot thể hiện phân bố F1-score per-class cho mỗi model — giúp thấy model nào đồng đều hơn qua các breeds.

In [ ]:
if all_eval_results:
    per_class_f1_data = []
    per_class_labels = []

    for model_name, results in all_eval_results.items():
        y_true = results["test_y_true"]
        y_pred = results["test_y_pred"]
        f1_per_class = f1_score(
            y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
        )
        per_class_f1_data.append(f1_per_class)
        per_class_labels.append(model_name)

    fig, ax = plt.subplots(figsize=(max(8, len(per_class_labels) * 2.5), 6))
    bp = ax.boxplot(
        per_class_f1_data,
        labels=per_class_labels,
        patch_artist=True,
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "red", "markersize": 6},
    )

    colors = ["#4c72b0", "#55a868", "#dd8452", "#c44e52"]
    for patch, color in zip(bp["boxes"], colors[: len(bp["boxes"])]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_ylabel("Per-class F1-score")
    ax.set_title("Per-class F1 Distribution (test set)", fontsize=14)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    for label, data in zip(per_class_labels, per_class_f1_data):
        print(
            f"{label:30s} | mean={np.mean(data):.4f} | "
            f"median={np.median(data):.4f} | min={np.min(data):.4f} | max={np.max(data):.4f}"
        )

## 11. Top-10 breeds khó nhất và dễ nhất (test set)

Dựa trên F1-score per-class của model có test accuracy cao nhất.

In [ ]:
if all_eval_results:
    best_model_name = max(
        all_eval_results, key=lambda m: all_eval_results[m]["test"]["accuracy"]
    )
    y_true = all_eval_results[best_model_name]["test_y_true"]
    y_pred = all_eval_results[best_model_name]["test_y_pred"]
    f1_per_class = f1_score(
        y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
    )

    sorted_indices = np.argsort(f1_per_class)

    print(f"\nBest model (by test accuracy): {best_model_name}\n")

    print("Top-10 EASIEST breeds (highest F1):")
    for rank, idx in enumerate(sorted_indices[-10:][::-1], 1):
        print(f"  {rank:2d}. {breed_names[idx]:30s} F1={f1_per_class[idx]:.4f}")

    print("\nTop-10 HARDEST breeds (lowest F1):")
    for rank, idx in enumerate(sorted_indices[:10], 1):
        print(f"  {rank:2d}. {breed_names[idx]:30s} F1={f1_per_class[idx]:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    easiest_idx = sorted_indices[-10:][::-1]
    axes[0].barh(
        range(10),
        f1_per_class[easiest_idx],
        color="#55a868",
        edgecolor="white",
    )
    axes[0].set_yticks(range(10))
    axes[0].set_yticklabels([breed_names[i] for i in easiest_idx], fontsize=9)
    axes[0].set_xlabel("F1-score")
    axes[0].set_title(f"Top-10 Easiest Breeds ({best_model_name})", fontsize=12)
    axes[0].set_xlim(0, 1.05)
    axes[0].invert_yaxis()

    hardest_idx = sorted_indices[:10]
    axes[1].barh(
        range(10),
        f1_per_class[hardest_idx],
        color="#c44e52",
        edgecolor="white",
    )
    axes[1].set_yticks(range(10))
    axes[1].set_yticklabels([breed_names[i] for i in hardest_idx], fontsize=9)
    axes[1].set_xlabel("F1-score")
    axes[1].set_title(f"Top-10 Hardest Breeds ({best_model_name})", fontsize=12)
    axes[1].set_xlim(0, 1.05)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

## 12. Lưu kết quả đánh giá ra file JSON

In [ ]:
EVAL_OUTPUT_PATH = HISTORY_DIR / "week10_evaluation_results.json"

export_data = {}
for model_name, results in all_eval_results.items():
    model_export = {"params": results["params"], "best_epoch": results["best_epoch"]}
    for split_name in ["train", "val", "test"]:
        m = results[split_name]
        model_export[split_name] = {
            "accuracy": m["accuracy"],
            "precision_macro": m["precision_macro"],
            "precision_weighted": m["precision_weighted"],
            "recall_macro": m["recall_macro"],
            "recall_weighted": m["recall_weighted"],
            "f1_macro": m["f1_macro"],
            "f1_weighted": m["f1_weighted"],
        }
    export_data[model_name] = model_export

EVAL_OUTPUT_PATH.write_text(
    json.dumps(export_data, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved evaluation results to: {EVAL_OUTPUT_PATH}")

## 13. Tổng kết

- Báo cáo này đánh giá tất cả các models có checkpoint sẵn trên cả 3 tập dữ liệu: **train, val, test**
- Mỗi model được đo lường với **7 chỉ số**: accuracy, precision (macro/weighted), recall (macro/weighted), F1 (macro/weighted)
- **Confusion matrix** giúp nhận diện các cặp giống dễ nhầm lẫn
- **Per-class F1 distribution** cho thấy mức độ đồng đều của model qua 120 giống chó

### Gợi ý bước tiếp theo

- Phân tích sâu hơn các cặp giống hay nhầm lẫn (nhóm theo họ terrier, họ spaniel, ...)
- Thử **TTA (Test-Time Augmentation)** để cải thiện kết quả inference
- Ensemble nhiều models (ResNet50 + EfficientNet-B0) và đánh giá lại